# Thomas Frank's Reign at Tottenham (25/26)

## Importing the Relevant Libraries

In [1]:
from bs4 import BeautifulSoup
import requests

import pandas as pd
import numpy as np
import itertools as it

## Creating the Dataframes for Thomas Frank Results

In [2]:
# Dataframe Columns that Will Store Stats about Each of Thomas Frank's Premier League Games in dataframe 'frank_prem_table'

prem_data = {
    "Game": [],
    "Home Team": [],
    "Away Team": [],
    "Home xG": [],
    "Away xG": [],
    "Home PPDA": [],
    "Away PPDA": [],
    "Home xPts": [],
    "Away xPts": [],
    "Home Yellows": [],
    "Away Yellows": [],
    "Home Reds": [],
    "Away Reds": [],
    "Home Penalties": [],
    "Away Penalties": [],
    "Home Possession": [],
    "Away Possession": [],
    "Home Passes Attempted": [],
    "Away Passes Attempted": [],
    "Home Passes Completed": [],
    "Away Passes Completed": [],
    "Home Pass Accuracy": [],
    "Away Pass Accuracy": [],
    "Home Shots on Target": [],
    "Away Shots on Target": [],
    "Home Shots off Target": [],
    "Away Shots off Target": [],
    "Home Shots inside Box": [],
    "Away Shots inside Box": [],
    "Home Shots outside Box": [],
    "Away Shots outside Box": [],
    "Home Total Shots": [],
    "Away Total Shots": [],
    "Home Corners": [],
    "Away Corners": [],
    "Home Fouls": [],
    "Away Fouls": [],
	"Home Offsides": [],
	"Away Offsides": [],
    "HT Home Goals": [],
    "HT Away Goals": [],
    "HT Score": [],
    "FT Home Goals": [],
    "FT Away Goals": [],
    "FT Score": [],
	"HT Result": [],
	"FT Result": [],
    "HT Points": [],
	"FT Points": []
}


frank_prem_table = pd.DataFrame(prem_data)

# Dataframe Columns that Will Store Stats about What Results Spurs Get after Coming from a Losing Position in dataframe 'match_position_table'

match_position_data = {
    "Match Position": [],
    "Wins": [],
    "Draws": [],
    "Losses": []
}

match_position_table = pd.DataFrame(match_position_data)
match_position_table.loc[len(match_position_table)] = ['Losing', 0, 0, 0]


# Dataframe Columns that Will Store the Segments in which a Goal was Scored/Conceded for Tottenham Separateley Home AND Away

home_away_goal_data = {
  "Minute Interval": [],
  "GF Home": [],
  "GF Away": [],
  "GA Home": [],
  "GA Away": []
}


frank_home_away_goal_table = pd.DataFrame(home_away_goal_data)


# Dataframe Columns that Will Store the Segments in which a Goal was Scored/Conceded for Tottenham COMBINED Home AND Away

overall_goal_data = {
  "Minute Interval": [],
  "GF": [],
  "GA": []
}

frank_overall_goal_table = pd.DataFrame(overall_goal_data)

# Adding the 'Minute Interval' columns to the 'frank_overall_goal_table' and 'frank_home_away_goal_table'
# so that the goals that were scoered/conceded in all of Thomas Frank's prem games is segmented properly

intervals_list = ["1-15", "16-30", "31-45", "46-60", "61-75", "76-90"]

for i in range(0, len(intervals_list)):
    frank_overall_goal_table.loc[len(frank_overall_goal_table)] = [intervals_list[i], 0, 0]
    frank_home_away_goal_table.loc[len(frank_home_away_goal_table)] = [intervals_list[i], 0, 0, 0, 0]


## Spurs Understat Prem Codes, Football Webpages Prem Codes, Other Segment Stats List, HT Scores and Prem Position Each Game Until Sacking

In [3]:
# Understat Prem Codes to Access Each Game's Page Data

frank_prem_codes = [28782, 28789, 28799, 28814, 28821, 28835, 28844, 28856, 
                    28863, 28876, 28886, 28889, 28906, 28916, 28926, 28935, 
                    28946, 28952, 28959, 28969, 28979, 28996, 29001, 29016, 29025, 29036]

# Football Webpage Prem Codes to Access Each Game's Page Data

fb_webpages_codes = ['tottenham-hotspur/burnley/529659', 'manchester-city/tottenham-hotspur/529671',
                'tottenham-hotspur/afc-bournemouth/529682', 'west-ham-united/tottenham-hotspur/529693',
                'brighton-and-hove-albion/tottenham-hotspur/529696', 'tottenham-hotspur/wolverhampton-wanderers/555161',
                'leeds-united/tottenham-hotspur/529720', 'tottenham-hotspur/aston-villa/553798',
                'everton/tottenham-hotspur/553805', 'tottenham-hotspur/chelsea/529752', 
                'tottenham-hotspur/manchester-united/529762', 'arsenal/tottenham-hotspur/556372', 
                'tottenham-hotspur/fulham/529782', 'newcastle-united/tottenham-hotspur/558102', 
                'tottenham-hotspur/brentford/529802', 'nottingham-forest/tottenham-hotspur/558108', 
                'tottenham-hotspur/liverpool/529822', 'crystal-palace/tottenham-hotspur/558677', 
                'brentford/tottenham-hotspur/558680', 'tottenham-hotspur/sunderland/558686', 
                'afc-bournemouth/tottenham-hotspur/529854', 'tottenham-hotspur/west-ham-united/529872',
                'burnley/tottenham-hotspur/529877', 'tottenham-hotspur/manchester-city/559426', 
                'manchester-united/tottenham-hotspur/529901', 'tottenham-hotspur/newcastle-united/560337']

# List that Stores the Other Key Match Stat Segments Which Helps Automate Collection

other_stats_list =['fwp-card yellow-card', 'fwp-card red-card', 
                     'pen)', '</th><th>Possession</th><th>', 
                     '</th><th>Passes attempted</th><th>', '</th><th>Passes completed</th><th>', 
                     '</th><th>Pass accuracy</th><th>', '</th><th>Shots on target</th><th>', 
                     '</th><th>Shots off target</th><th>', '</th><th>Shots inside box</th><th>', 
                     '</th><th>Shots outside box</th><th>', '</th><th>Total shots</th><th>', 
                     '</th><th>Corner kicks</th><th>', '</th><th>Fouls conceded</th><th>', 
                     '</th><th>Offsides</th><th>']

# Half Time Scores for Each Prem Game


frank_half_time_list = ["1 - 0", "0 - 2", "0 - 1", "0 - 0", "2 - 1", 
                        "0 - 0", "1 - 1", "1 - 1", "0 - 2", "0 - 1", 
                        "0 - 1", "2 - 0", "0 - 2", "0 - 0", "2 - 0", 
                        "1 - 0", "0 - 0", "0 - 1", "0 - 0", "1 - 0", 
                        "2 - 1", "0 - 1", "1 - 1", "0 - 2", "1 - 0", "0 - 1"]

# Spurs Prem Position after Each Game

frank_prem_position = ['3', '2', '4', '3', '3', 
                       '4', '3', '6', '3', '6', 
                       '5', '9', '12', '11', '11', 
                       '11', '14', '11', '12', '13', 
                       '14', '14', '14', '14', '15', '16']


## Method to Categorise Goal Segments When Tottenham/Opposition Scored

In [4]:
# The Method 'goal_segment' Accepts Four Parameters. These Are:

# 'gm' - Minute the Goal Has Been Scored
# 'col_pos' - Team who Scored the Goal in the Game and Type (Spurs Scoring at Home, Spurs Scoring Away, Opposition scoring at Tottenham Hotspur Stadium, Opposition Scoring at THEIR Home)
# 'ge' - Score When a Goal Has Been Scored in a Match
# 'hts' - Half Time Score of a Game

def goal_segment(gm, col_pos, ge, hts):
    
    # Set 'home_goal_counter' and 'away_goal_counter' to Global as We Need to Update These Variables
    # in Both the 'goal_segment' and 'collect_results' Methods
    global home_goal_counter
    global away_goal_counter

    # Tottenham Scored at Home (Tottenham Hotspur Stadium)
    if col_pos == 1:
        col_name = 'GF Home'
        
    # Tottenham Scored AWAY from Home
    if col_pos == 2:
        col_name = 'GF Away'
        
    # Opposition Scored at Tottenham Hotspur Stadium
    if col_pos == 3:
        col_name = 'GA Home'
        
    # Opposition Scored at THEIR Home
    if col_pos == 4:
        col_name = 'GA Away'
        
    # Team Scored Within the First 15 minutes
    if int(gm) > 0 and int(gm) <= 15:
        
        # Home Team Scored
        if int(home_goal_counter) != int(ge[0]):
            home_goal_counter = int(ge[0])
            old_value = frank_home_away_goal_table.iat[0, col_pos]
            frank_home_away_goal_table.at[0, col_name]= old_value + 1
            
        # Away Team Scored
        elif int(away_goal_counter) != int(ge[4]):
            away_goal_counter = int(ge[4])
            old_value = frank_home_away_goal_table.iat[0, col_pos]
            frank_home_away_goal_table.at[0, col_name]= old_value + 1
    
    
    # Team Scored Within the 16-30 minute interval
    if int(gm) >= 16 and int(gm) <= 30:
        
        # Home Team Scored
        if int(home_goal_counter) != int(ge[0]):
            home_goal_counter = int(ge[0])
            old_value = frank_home_away_goal_table.iat[1, col_pos]
            frank_home_away_goal_table.at[1, col_name]= old_value + 1
            
        # Away Team Scored
        elif int(away_goal_counter) != int(ge[4]):
            away_goal_counter = int(ge[4])
            old_value = frank_home_away_goal_table.iat[1, col_pos]
            frank_home_away_goal_table.at[1, col_name]= old_value + 1
            
    # Team Scored Within the 31-45 minute interval
    if int(gm) >= 31 and int(gm) <= 45:
    
        # Home Team Scored
        if int(home_goal_counter) != int(ge[0]):
            home_goal_counter = int(ge[0])
            old_value = frank_home_away_goal_table.iat[2, col_pos]
            frank_home_away_goal_table.at[2, col_name]= old_value + 1
            
        # Away Team Scored
        elif int(away_goal_counter) != int(ge[4]):
            away_goal_counter = int(ge[4])
            old_value = frank_home_away_goal_table.iat[2, col_pos]
            frank_home_away_goal_table.at[2, col_name]= old_value + 1
    
            
    # Team Scored Within the 46-60 minute interval
    if int(gm) >= 46 and int(gm) <= 60:
    
        # Home Team Scored
        if int(home_goal_counter) != int(ge[0]):
            home_goal_counter = int(ge[0])

            # Home Team Scored in Second Half
            if int(home_goal_counter) > int(hts[0]):
                old_value = frank_home_away_goal_table.iat[3, col_pos]
                frank_home_away_goal_table.at[3, col_name]= old_value + 1
            
            # Home Team Scored in First Half ADDED TIME
            else:
                old_value = frank_home_away_goal_table.iat[2, col_pos]
                frank_home_away_goal_table.at[2, col_name]= old_value + 1                                    
            
        # Away Team Scored
        elif int(away_goal_counter) != int(ge[4]):
            
            away_goal_counter = int(ge[4])    
            
            # Away Team Scored in Second Half
            if int(away_goal_counter) > int(hts[4]):
                old_value = frank_home_away_goal_table.iat[3, col_pos]
                frank_home_away_goal_table.at[3, col_name]= old_value + 1
    
            # Away Team Scored in First Half ADDED TIME
            else:
                old_value = frank_home_away_goal_table.iat[2, col_pos]
                frank_home_away_goal_table.at[2, col_name]= old_value + 1
    
    
    # Team Scored Within the 61-75 minute interval
    if int(gm) >= 61 and int(gm) <= 75:
        
        # # Home Team Scored
        if int(home_goal_counter) != int(ge[0]):
            home_goal_counter = int(ge[0])
            old_value = frank_home_away_goal_table.iat[4, col_pos]
            frank_home_away_goal_table.at[4, col_name]= old_value + 1
            
        # Away Team Scored
        elif int(away_goal_counter) != int(ge[4]):
            away_goal_counter = int(ge[4])
            old_value = frank_home_away_goal_table.iat[4, col_pos]
            frank_home_away_goal_table.at[4, col_name]= old_value + 1
    
    # Team Scored Within the 76-90 minute interval
    if int(gm) >= 76:
        
        # Home Team Scored
        if int(home_goal_counter) != int(ge[0]):
            home_goal_counter = int(ge[0])
            old_value = frank_home_away_goal_table.iat[5, col_pos]
            frank_home_away_goal_table.at[5, col_name]= old_value + 1
            
        # Away Team Scored
        elif int(away_goal_counter) != int(ge[4]):
            away_goal_counter = int(ge[4])
            old_value = frank_home_away_goal_table.iat[5, col_pos]
            frank_home_away_goal_table.at[5, col_name]= old_value + 1

## MAIN Method to Collect Results and Add to Relevant Dataframes

In [5]:
# Website Where We Are Scraping the Main Tottenham Premier League Data from For Each Game.
understat_website = 'https://understat.com/match/'

# Website Where We Are Scraping the Other Tottenham Premier League Data from For Each Game.
fb_webpages_site = 'https://www.footballwebpages.co.uk/match/2025-2026/premier-league/'


tag_attributes = ['col-12 col-sm-6 col-lg-3 order-lg-1 home-line-up', 'col-12 col-sm-6 col-lg-3 order-lg-3 away-line-up', 
                  'team away-team has-crest', 'team home-team has-crest']

global tag
tag = "div"


# The Method 'collect_results' Accepts Three Parameters. These Are:

# 'game_no' - Game Number the Match is.
# 'codes_1' - List of Understat Codes of Thomas Frank 25/26 Prem Games Until His Sacking
# 'codes_2' - List of Football Webpages Codes of Thomas Frank 25/26 Prem Games Until His Sacking
# 'other_stats' - List of Oher Measured Stats for Each Game
# 'half_time' - List of Scores at Half-Time for Each Game
# 'prem_position' - List of Tottenham Position in Table After Each Game
# 'prem_table' - Dataframe Used to Store Each of the Games Data


def collect_results(game_no, codes_1, codes_2, other_stats, half_time, prem_position, prem_table):

    global tag
    tag = "div"

    # Variable 'half_time_counter' Which Obtains the Half Time Result for Each Game From the 'half_time'
    # Starts at -1 as the Method's Main For Loop Increments the Value by 1 to Start at the First Half Time Result
    half_time_counter = -1

    # Main For Loop Which Goes Through Each Understat and Football Webpage Prem Code of Thomas Frank's Tenure.
    # Will Allow for the Relevant Data to be Scraped and Added to the Correct Dataframes.
    for i, j in zip(codes_1, codes_2):

        # Creating the Correct Understat URL for Each Thomas Frank Game then Using the BeautifulSoup Package to Extract the Data
        understat_url = understat_website + str(i)
        understat_page = requests.get(understat_url)
        understat_soup = BeautifulSoup(understat_page.text, 'html')
        
        # Creating the Correct Football Webpages URL for Each Thomas Frank Game then Using the BeautifulSoup Package to Extract the Data
        fb_webpages_url = fb_webpages_site + str(j)
        fb_webpages_page = requests.get(fb_webpages_url)
        fb_webpages_soup = BeautifulSoup(fb_webpages_page.text, 'html')

        # Finding and Storing the Key Stats from Each Individual Understat Page/Match Using BeautifulSoup By Searching With the 'progress-value' Div Tag
        # Stored in a List called 'match_table'
        match_table = understat_soup.find_all("div", "progress-value")

        # Finding and Storing the Away Team Stats By Searching With the 'timeline-item timeline-item-right' Div Tag
        # Stored in a List called 'away_event_table'
        # Converting the List 'away_event_table' to a String 'away_event_count' so that the Away Team Stat Occurances Can Be Counted to Calculate Home Team Stat Occurances
        away_event_table = understat_soup.find_all("div", "timeline-item timeline-item-right")
        away_event_count = str(away_event_table)

        # Finding and Storing the Key Stats from Each Individual Football Webpage/Match Using BeautifulSoup By Searching With the 'tab-content' Div Tag
        # Converting the List 'other_stats_table' to a String 'other_stats_string' so that the Other Stats Can Be split in 2 (Home and Away) and Extracted
        other_stats_table = fb_webpages_soup.find_all("div", "tab-content")
        other_stats_string = str(other_stats_table)

        # Counting Each Game so that the For Loop Goes Through to Put Into the Main Dataframe (prem_table)
        game_no += 1

        # The Home and Away Team Playing the Match
        home_team = match_table[0].text
        away_team = match_table[1].text

        # 'stat_counter' Counting Each Stat so that the INNER For Loop Goes Through All Stat Segments in the 'other_stats' list
        stat_counter = 0

        # List 'other_stats_values' which will Store the Values of the Home and Away Teams Stats like Yellows, Reds, etc
        other_stats_values = []

        # INNER For Loop That Goes Through All Stat Segments in the 'other_stats' list
        for k in other_stats:

            # Adding 1 to 'stat_counter' so that the Stat Values can be Extracted Correctly
            stat_counter += 1

            # Searching for Data Using Div Tag for First 2 Stats (Yellow and Red Cards)
            if stat_counter != 3:
                tag = "div"

            # Searching for Data Using Li Tag for Third Stat (Pens)
            # Reversing the 'tag_attributes' List to Place The Last Two Li Tag Attributes as the First Two
            if stat_counter == 3:
                tag_attributes.reverse()
                tag = "li"

            # First Three Stat Values are Yellow, Red and Penalty Stats from Understat Website
            if stat_counter <= 3:
                
                # INNER For Loop to Grab First Two Tag Attributes from List ' tag_attributes'
                # Then Using BeautifulSoup to Create a List then Convert to String to Count for Occurances of Yellows, Reds and Pens
                # Adding Stat Occurance to List 'other_stats_values'
                for attribute in it.islice(tag_attributes, 2):
        
                    other_stats_yrp_table = fb_webpages_soup.find_all(tag, attribute)
                    other_stats_yrp_string = str(other_stats_yrp_table)
                    stat_value = other_stats_yrp_string.count(k)
                    other_stats_values.append(stat_value)


            
            # If the Stat Segment is For Pens Then Re-Reverse the 'tag_attributes' to it's Original Order for Future Use
            if stat_counter == 3:
                tag_attributes.reverse()

            # Other Stat Segments are from the Football Webpages Site
            if stat_counter > 3:

                # 'segment_count Splits the String 'other_stats_string' so that the Stats Can Be Distinguished for Home and Away Teams
                segment_count = other_stats_string.split(k, 2)

                # Stat Segments Between 4 and 7 Are Percentage Based
                if stat_counter <= 7:
                
                    home_stat = segment_count[0][-3:]
                    away_stat = segment_count[1][:3]

                # Other Stat Segments are Number Based
                else:

                    # Getting the Last/First Two Characters of String 'segment_count' for Home and Away Occurances

                    home_stat = segment_count[0][-2:]
                    away_stat = segment_count[1][:2]

                    # If the Stat Value is a Single Digit then the Non-Numberical Character is Removed
        
                    if '>' in home_stat:
                        
                        home_stat = segment_count[0][-1:]
                        
                    if '<' in away_stat:
                        
                        away_stat = segment_count[1][:1]

                # Adding the Home and Away Stat Segment Values to the List 'other_stats_values'
                other_stats_values.append(home_stat)
                other_stats_values.append(away_stat)
                    

        # Getting the Amount of Home and Away goals Scored in a Match at FULL TIME from the BeautifulSoup 'match_table' list
        ft_home_goals = match_table[5].text
        ft_away_goals = match_table[6].text

        # Adding 1 to the 'half_time_counter' Variable to get the Correct Correspoding Half-Time Score for each Match from the 'half_time'    
        half_time_counter += 1
        half_time_score = half_time[half_time_counter]

        # Getting the Amount of Home and Away goals Scored in a Match at HALF TIME
        ht_home_goals = half_time_score[0]
        ht_away_goals = half_time_score[4]

        # Adding the Key Stats from Each Game into the Main Dataframe (prem_table)
        prem_table.loc[len(prem_table)] = [game_no,
                                           home_team, 
                                           away_team, 
                                           float(match_table[7].text), 
                                           float(match_table[8].text), 
                                           float(match_table[15].text), 
                                           float(match_table[16].text), 
                                           float(match_table[17].text),  
                                           float(match_table[18].text),
                                           int(other_stats_values[0]),
                                           int(other_stats_values[1]),
                                           int(other_stats_values[2]),
                                           int(other_stats_values[3]),
                                           int(other_stats_values[4]),
                                           int(other_stats_values[5]),
                                           other_stats_values[6],
                                           other_stats_values[7],
                                           int(other_stats_values[8]),
                                           int(other_stats_values[9]),
                                           int(other_stats_values[10]),
                                           int(other_stats_values[11]),
                                           other_stats_values[12],
                                           other_stats_values[13],
                                           int(other_stats_values[14]),
                                           int(other_stats_values[15]),
                                           int(other_stats_values[16]),
                                           int(other_stats_values[17]),
                                           int(other_stats_values[18]),
                                           int(other_stats_values[19]),
                                           int(other_stats_values[20]),
                                           int(other_stats_values[21]),
                                           int(other_stats_values[22]),
                                           int(other_stats_values[23]),
                                           int(other_stats_values[24]),
                                           int(other_stats_values[25]),
                                           int(other_stats_values[26]),
                                           int(other_stats_values[27]),
                                           int(other_stats_values[28]),
                                           int(other_stats_values[29]),
                                           int(ht_home_goals), 
                                           int(ht_away_goals),
                                           half_time_score,
                                           int(ft_home_goals),
                                           int(ft_away_goals),
                                           ft_home_goals + " - " + ft_away_goals,
                                           "",
                                           "",
                                           0,
                                           0]



        # Character Used by understat to Signal What Minute the Goal was Scored by Either the Home or Away team
        minute_character = "'"

        # Setting the Home and Away Goal Counters to 0
        # These Counters Signify Whether the Home or Away Team has Scored and that it has been Recorded in the Dataframe 'frank_home_away_goal_table'
        global home_goal_counter
        home_goal_counter = 0
        global away_goal_counter
        away_goal_counter = 0

        # Setting the Variable 'losing_state' to 0
        # This Variable Helps Indicate When If Spurs Were Ever Trailing (Losing) in a Game

        losing_state = 0

        # Setting Variable 'match_state' to blank.
        # Will Be Used To Determine What Result Spurs Got IF They Were Losing in a Match
        match_state = ""

        # So that the Correct Match Result Can Be Determined Two Variables 'tgn' and 'ogn' are used
        # 'tgn' - Tottenham Goal Number. IF Spurs Play at Home then the Value is 0. IF Away then Value is 4
        # 'ogn' - Opposition Goal Number. IF Opposition Plays at Tottenham Stadium then the Value is 4. IF they are Home then Value is 0

        if home_team == "Tottenham":
            tgn = 0
            ogn = 4

        else:
            tgn = 4
            ogn = 0
        
        # Finding How the Goal Score Lines Change During Each Match using the Span Tag 'timeline-match-score' 
        goal_table = understat_soup.find_all("span", "timeline-match-score")
    
        # Finding Key Match Events from Each Individual Match
        match_event_table = understat_soup.find_all("div", "timeline-item")
        
        #INNER For loop to Run Through All Events for Each Match and Properly Format then Store in 'match_event' Variable
        for l in range(0, len(match_event_table)):
        
            match_event = match_event_table[l].text
            match_event = match_event.strip("\n")
            match_event = match_event.replace("\n", "")
        
            # SECOND INNER For loop to Run Through when Goals Occured and Match to the Correct Match Event
            for m in range(0, len(goal_table)):
            
                # Variable 'goal_event' Represents the Score when a Goal is Scored in a Game
                goal_event = goal_table[m].text
                
                # Variable 'match_event_goal' Searches if the CURRENT Match Event Matches the CURRENT Goal Scored Event
                match_event_goal = match_event.find(goal_event)
            
                # This Means that the Match Event is a Goal
                if match_event_goal != -1:
                    
                    # Find Out the Minute the Goal was Scored In and Store it in 'goal_minute' and Add 1 as Understat Did Not
                    # Correctly Store the Correct Goal Minutes
                    find_minute = match_event.find(minute_character)
                    goal_minute = match_event[:find_minute]
                    goal_minute = int(goal_minute) + 1
                                        
                    # Game at Tottenham Hotspur Stadium (Home)
                    if home_team == "Tottenham":
                            
                        # Tottenham Scored
                        if int(home_goal_counter) != int(goal_event[0]):
                            goal_segment(goal_minute, 1, goal_event, half_time_score)
                            
                        # Away Team Scored
                        elif int(away_goal_counter) != int(goal_event[4]):
                            goal_segment(goal_minute, 3, goal_event, half_time_score)
                                
                    # Game at Away Stadium (NOT TOTTENHAM)
                    if home_team != "Tottenham":

                        # Opposition Scored
                        if int(home_goal_counter) != int(goal_event[0]):
                            goal_segment(goal_minute, 4, goal_event, half_time_score)
                            
                        # Tottenham Scored
                        elif int(away_goal_counter) != int(goal_event[4]):
                            goal_segment(goal_minute, 2, goal_event, half_time_score)

                    # If Home and Away Goals Are Equal to Each Other in Particular Point in Time of Match then it is Currently a DRAW
                    if int(goal_event[tgn]) == int(goal_event[ogn]):
                        match_state = "DRAW"

                    # If Spurs Home Goals are LESS than Opposition's Goals in Particular Point in Time of Match then it is Currently a LOSS
                    # Variable 'losing_state' is Set to 1 to Signify Spurs are Trailing/Losing in a Match
                    if int(goal_event[tgn]) < int(goal_event[ogn]):
                        losing_state = 1
                        match_state = "LOSS"

                    # If Spurs Home Goals are MORE than Opposition's Goals in Particular Point in Time of Match then it is Currently a WIN
                    if int(goal_event[tgn]) > int(goal_event[ogn]):
                        match_state = "WIN"


        # IF and ELSE Statements Check Values of Variables 'losing_state' and 'match_state'
        # IF Variable 'losing_state' is Equal to 1 then Spurs Were Trailing and the Vairable 'match_state' Can Be Examined
        # To Determine What Result Spurs Got After Losing In a Match By Updating the 'match_position_table' Dataframe

        if losing_state == 1 and match_state == "WIN":
            old_value = match_position_table.iat[0, 1]
            match_position_table.at[0, 'Wins']= old_value + 1
        
        elif losing_state == 1 and match_state == "DRAW":
            old_value = match_position_table.iat[0, 2]
            match_position_table.at[0, 'Draws']= old_value + 1
            
        elif losing_state == 1 and match_state == "LOSS":
            old_value = match_position_table.iat[0, 3]
            match_position_table.at[0, 'Losses']= old_value + 1


    # Result Conditions. Need to Work this Out to Add to Main Dataframe 'prem_table'
    # This Will Tell us Whether Tottenham Won the Game or Not

    # 'ms' list Represents Match Stage HT (Half Time) and FT (Full Time)
    # Important to Distinguish what the Results were for Tottenham at HT and FT
    ms = ['HT', 'FT']
    
    # Inner For Loop Which Goes Through the List 'ms'
    for n in range(len(ms)):

        # Getting Name of Dataframe Column for Home Goals of Specific Match Stage
        h_goal = ms[n] + ' Home Goals'
        # Getting Name of Dataframe Column for Away Goals of Specific Match Stage
        a_goal = ms[n] + ' Away Goals'

        ms_result = ms[n] + ' Result'

        ms_points = ms[n] + ' Points'

    # There are a total of 5 conditions. These are:
    
        # If Home Goals is EQUAL to Away Goals Scored then the result is a DRAW
        # If the Home Goals are GREATER than Away Goals Scored and the Home Team is Tottenham then result is a WIN
        # If the Away Goals are GREATER than Home Goals Scored and the Away Team is Tottenham then result is a WIN
        # If the Away Goals are GREATER than Home Goals Scored and the Home Team is Tottenham then result is a LOSS
        # If the Home Goals are GREATER than Away Goals Scored and the Away Team is Tottenham then result is a LOSS
        
        list_conditions = [
            (prem_table[h_goal] == prem_table[a_goal]),
            (prem_table[h_goal] > prem_table[a_goal]) & (prem_table['Home Team'] == "Tottenham"),
            (prem_table[h_goal] < prem_table[a_goal]) & (prem_table['Away Team'] == "Tottenham"),
            (prem_table[h_goal] < prem_table[a_goal]) & (prem_table['Home Team'] == "Tottenham"),
            (prem_table[h_goal] > prem_table[a_goal]) & (prem_table['Away Team'] == "Tottenham")
            ]
    
        # Spurs Result Values (Draw, Win, Loss) Based on Conditions
        list_results = ['Draw', 'Win', 'Win', 'Loss', 'Loss']

        # Spurs Result Points (Draw = 1, Win = 3, Loss = 0) Based on Conditions
        list_points = [1, 3, 3, 0, 0]
    
        # New Columns to Assign Correct Results to Tottenham Games
        prem_table[ms_result] = np.select(list_conditions, list_results, default='Other')

        # New Columns to Assign Correct Pointss to Tottenham Games
        prem_table[ms_points] = np.select(list_conditions, list_points, default=0)

    # Column 'Table Position' equal to list 'prem_position' which represents what position Tottenham were in table after EACH game played
    prem_table['Table Position'] = prem_position
    

## Running the Main Method to Collect Results

In [6]:
collect_results(0, frank_prem_codes, fb_webpages_codes, other_stats_list, frank_half_time_list, frank_prem_position, frank_prem_table)

## Correctly Formatting All Tottenham Results

### Tottenham Home Results

In [7]:
frank_home = frank_prem_table[frank_prem_table['Home Team'] == 'Tottenham']

frank_prem_table_home = frank_home.rename(columns={'Home xG': 'Tottenham xG', 
                                      'Away xG': 'Opposition xG',
                                      'Home PPDA': 'Tottenham PPDA',
                                      'Away PPDA': 'Opposition PPDA',
                                      'Home xPts': 'Tottenham xPts',
                                      'Away xPts': 'Opposition xPts',
                                      'Home Yellows': 'Tottenham Yellows',
                                      'Away Yellows': 'Opposition Yellows',
                                      'Home Reds': 'Tottenham Reds',
                                      'Away Reds': 'Opposition Reds',
                                      'Home Penalties': 'Tottenham Penalties',
                                      'Away Penalties': 'Opposition Penalties',
                                      'Home Possession': 'Tottenham Possession',
                                      'Away Possession': 'Opposition Possession',
                                      'Home Passes Attempted': 'Tottenham Passes Attempted',
                                      'Away Passes Attempted': 'Opposition Passes Attempted',
                                      'Home Passes Completed': 'Tottenham Passes Completed',
                                      'Away Passes Completed': 'Opposition Passes Completed',
                                      'Home Pass Accuracy': 'Tottenham Pass Accuracy',
                                      'Away Pass Accuracy': 'Opposition Pass Accuracy',
                                      'Home Shots on Target': 'Tottenham Shots on Target',
                                      'Away Shots on Target': 'Opposition Shots on Target',
                                      'Home Shots off Target': 'Tottenham Shots off Target',
                                      'Away Shots off Target': 'Opposition Shots off Target',
                                      'Home Shots inside Box': 'Tottenham Shots inside Box',
                                      'Away Shots inside Box': 'Opposition Shots inside Box',
                                      'Home Shots outside Box': 'Tottenham Shots outside Box',
                                      'Away Shots outside Box': 'Opposition Shots outside Box',
                                      'Home Total Shots': 'Tottenham Total Shots',
                                      'Away Total Shots': 'Opposition Total Shots',
                                      'Home Corners': 'Tottenham Corners',
                                      'Away Corners': 'Opposition Corners',
                                      'Home Fouls': 'Tottenham Fouls',
                                      'Away Fouls': 'Opposition Fouls',
                                      'Home Offsides': 'Tottenham Offsides',
                                      'Away Offsides': 'Opposition Offsides',
                                      'HT Home Goals': 'HT Tottenham Goals',
                                      'HT Away Goals': 'HT Opposition Goals',
                                      'FT Home Goals': 'FT Tottenham Goals',
                                      'FT Away Goals': 'FT Opposition Goals'})

del frank_home

### Tottenham Away Results

In [8]:
frank_away = frank_prem_table[frank_prem_table['Away Team'] == 'Tottenham']

frank_prem_table_away = frank_away.rename(columns={'Home xG': 'Opposition xG', 
                                      'Away xG': 'Tottenham xG',
                                      'Home PPDA': 'Opposition PPDA',
                                      'Away PPDA': 'Tottenham PPDA',
                                      'Home xPts': 'Opposition xPts',
                                      'Away xPts': 'Tottenham xPts',
                                      'Home Yellows': 'Opposition Yellows',
                                      'Away Yellows': 'Tottenham Yellows',
                                      'Home Reds': 'Opposition Reds',
                                      'Away Reds': 'Tottenham Reds',
                                      'Home Penalties': 'Opposition Penalties',
                                      'Away Penalties': 'Tottenham Penalties',
                                      'Home Possession': 'Opposition Possession',
                                      'Away Possession': 'Tottenham Possession',
                                      'Home Passes Attempted': 'Opposition Passes Attempted',
                                      'Away Passes Attempted': 'Tottenham Passes Attempted',
                                      'Home Passes Completed': 'Opposition Passes Completed',
                                      'Away Passes Completed': 'Tottenham Passes Completed',
                                      'Home Pass Accuracy': 'Opposition Pass Accuracy',
                                      'Away Pass Accuracy': 'Tottenham Pass Accuracy',
                                      'Home Shots on Target': 'Opposition Shots on Target',
                                      'Away Shots on Target': 'Tottenham Shots on Target',
                                      'Home Shots off Target': 'Opposition Shots off Target',
                                      'Away Shots off Target': 'Tottenham Shots off Target',
                                      'Home Shots inside Box': 'Opposition Shots inside Box',
                                      'Away Shots inside Box': 'Tottenham Shots inside Box',
                                      'Home Shots outside Box': 'Opposition Shots outside Box',
                                      'Away Shots outside Box': 'Tottenham Shots outside Box',
                                      'Home Total Shots': 'Opposition Total Shots',
                                      'Away Total Shots': 'Tottenham Total Shots',
                                      'Home Corners': 'Opposition Corners',
                                      'Away Corners': 'Tottenham Corners',
                                      'Home Fouls': 'Opposition Fouls',
                                      'Away Fouls': 'Tottenham Fouls',
                                      'Home Offsides': 'Opposition Offsides',
                                      'Away Offsides': 'Tottenham Offsides',
                                      'HT Home Goals': 'HT Opposition Goals',
                                      'HT Away Goals': 'HT Tottenham Goals',
                                      'FT Home Goals': 'FT Opposition Goals',
                                      'FT Away Goals': 'FT Tottenham Goals'})

del frank_away

### Tottenham ALL Results (Now in Correct Format)

In [9]:
del frank_prem_table

frank_prem_table = pd.concat([frank_prem_table_home, frank_prem_table_away])

del frank_prem_table_home
del frank_prem_table_away

frank_prem_table = frank_prem_table.sort_values('Game')

frank_prem_table

,Game,Home Team,Away Team,Tottenham xG,Opposition xG,Tottenham PPDA,Opposition PPDA,Tottenham xPts,Opposition xPts,Tottenham Yellows,...,HT Opposition Goals,HT Score,FT Tottenham Goals,FT Opposition Goals,FT Score,HT Result,FT Result,HT Points,FT Points,Table Position
0,1,Tottenham,Burnley,1.98,0.73,6.35,17.35,2.31,0.49,0,...,0,1 - 0,3,0,3 - 0,Win,Win,3,3,3
1,2,Manchester City,Tottenham,1.28,1.25,13.27,10.44,1.36,1.35,4,...,0,0 - 2,2,0,0 - 2,Win,Win,3,3,2
2,3,Tottenham,Bournemouth,0.17,2.12,5.20,7.15,0.16,2.73,2,...,1,0 - 1,0,1,0 - 1,Loss,Loss,0,0,4
3,4,West Ham,Tottenham,1.51,0.79,10.86,17.56,1.93,0.79,1,...,0,0 - 0,3,0,0 - 3,Draw,Win,1,3,3
4,5,Brighton,Tottenham,1.19,1.40,4.84,11.48,1.13,1.54,2,...,2,2 - 1,2,2,2 - 2,Loss,Draw,0,1,3
5,6,Tottenham,Wolverhampton Wanderers,0.82,1.16,16.93,9.50,0.99,1.67,3,...,0,0 - 0,1,1,1 - 1,Draw,Draw,1,1,4
6,7,Leeds,Tottenham,0.69,1.48,15.00,10.83,0.67,2.06,3,...,1,1 - 1,2,1,1 - 2,Draw,Win,1,3,3
7,8,Tottenham,Aston Villa,1.51,0.35,7.59,12.89,2.38,0.41,2,...,1,1 - 1,1,2,1 - 2,Draw,Loss,1,0,6
8,9,Everton,Tottenham,2.09,1.40,17.00,13.42,1.92,0.83,0,...,0,0 - 2,3,0,0 - 3,Win,Win,3,3,3
9,10,Tottenham,Chelsea,0.15,3.34,8.56,7.00,0.02,2.96,4,...,1,0 - 1,0,1,0 - 1,Loss,Loss,0,0,6


### Exporting ALL Tottenham Results as CSV

In [10]:
frank_prem_table.to_csv('frank_prem_table.csv', index=False)

## Viewing How Tottenham Scored/Conceded SEPARETELY Home and Away

In [11]:
frank_home_away_goal_table

,Minute Interval,GF Home,GF Away,GA Home,GA Away
0,1-15,2,1,5,1
1,16-30,2,2,0,2
2,31-45,1,6,5,7
3,46-60,3,4,2,2
4,61-75,4,1,2,1
5,76-90,4,6,4,6


### Exporting How Tottenham Scored/Conceded SEPARETELY as CSV

In [12]:
frank_home_away_goal_table.to_csv('frank_home_away_goal_table.csv', index=False)

## Viewing How Tottenham Scored/Conceded OVERALL

In [13]:
frank_overall_goal_table['GF'] = frank_home_away_goal_table['GF Home'] + frank_home_away_goal_table['GF Away']
frank_overall_goal_table['GA'] = frank_home_away_goal_table['GA Home'] + frank_home_away_goal_table['GA Away']

frank_overall_goal_table

,Minute Interval,GF,GA
0,1-15,3,6
1,16-30,4,2
2,31-45,7,12
3,46-60,7,4
4,61-75,5,3
5,76-90,10,10


### Exporting How Tottenham Scored/Conceded OVERALL as CSV

In [14]:
frank_overall_goal_table.to_csv('frank_overall_goal_table.csv', index=False)

## Viewing Tottenham Results After Being in a Losing Position

In [15]:
match_position_table

,Match Position,Wins,Draws,Losses
0,Losing,0,6,11


### Exporting Tottenham Results After LOSING POSITION as CSV

In [16]:
match_position_table.to_csv('match_position_table.csv', index=False)